# 00_config — 共通設定

全Notebookが参照する設定値を定義し、環境確認・パス検証を行う。

**モード切り替え**
| 変数 | 値 | 説明 |
|---|---|---|
| `DATA_SOURCE_MODE` | `existing` | 既存 keiba_ultimate.db を利用 |
| `DATA_SOURCE_MODE` | `simulation` | 実スクレイピングで sim DB を生成 |
| `FEATURE_SOURCE` | `existing` / `simulation` | 特徴量エンジニアリングの入力 DB |

In [1]:
# ── ブートストラップ ──────────────────────────────────────────
import sys
from pathlib import Path

# notebooks/ ディレクトリを sys.path に追加
_NB_DIR = Path().resolve()
if _NB_DIR.name != 'notebooks':
    _NB_DIR = _NB_DIR.parent
if str(_NB_DIR) not in sys.path:
    sys.path.insert(0, str(_NB_DIR))

from utils.nb_config import *
print_config()

  Keiba AI Notebook 設定
  ROOT           : C:\Users\yuki2\Documents\ws\keiba-ai-pro
  DATA_SOURCE    : existing
  FEATURE_SOURCE : existing
  DB (existing)  : C:\Users\yuki2\Documents\ws\keiba-ai-pro\keiba\data\keiba_ultimate.db (存在)
  DB (sim)       : C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\simulation\keiba_sim.db
  TRAIN          : 20200101 ～ 20241231
  TEST           : 20250101 ～ 20251231
  TARGET         : win
  MODELS_DIR     : C:\Users\yuki2\Documents\ws\keiba-ai-pro\python-api\models
  REPORTS_DIR    : C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\reports
  FEATURE_STORE  : C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\feature_store
  MODEL_STORE    : C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\model_store
  REPORT_STORE   : C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\data\report_store


## ユーザー設定（ここで変更する）

In [2]:
# ══════════════════════════════════════════════════════════
# ▼ 変更箇所: 必要に応じて書き換えてください
# ══════════════════════════════════════════════════════════

DATA_SOURCE_MODE = "existing"   # "existing" | "simulation"
FEATURE_SOURCE   = "existing"   # "existing" | "simulation"

TRAIN_START = "20130101"  # 学習期間開始 (YYYYMMDD)
TRAIN_END   = "20170101"  # 学習期間終了 (YYYYMMDD)
TEST_START  = "20170102"  # テスト期間開始 (YYYYMMDD)
TEST_END    = "20180505"  # テスト期間終了 (YYYYMMDD)

# TARGET        = "win"              # 単勝予測（2値分類: 1着 = 1）
# TARGET      = "place3"           # 複勝予測（2値分類: 3着以内 = 1）
TARGET      = "speed_deviation"  # 速度偏差予測（回帰）

CV_FOLDS      = 5
OPTUNA_TRIALS = 5










RANDOM_STATE  = 42

# simulation モード用スクレイプ期間
SIM_SCRAPE_START = "2026-01"   # YYYY-MM
SIM_SCRAPE_END   = "2026-02"   # YYYY-MM

print(f"DATA_SOURCE_MODE = {DATA_SOURCE_MODE}")
print(f"FEATURE_SOURCE   = {FEATURE_SOURCE}")
print(f"学習期間: {TRAIN_START} ～ {TRAIN_END}")
print(f"テスト期間: {TEST_START} ～ {TEST_END}")


DATA_SOURCE_MODE = existing
FEATURE_SOURCE   = existing
学習期間: 20130101 ～ 20170101
テスト期間: 20170102 ～ 20180505


## 環境確認

In [3]:
# ── 依存パッケージ確認 ──────────────────────────────────────
import importlib

_REQUIRED = [
    ("lightgbm",  "lightgbm"),
    ("pandas",    "pandas"),
    ("numpy",     "numpy"),
    ("sklearn",   "scikit-learn"),
    ("optuna",    "optuna"),
    ("matplotlib","matplotlib"),
    ("shap",      "shap"),
]
_OPTIONAL = [
    ("catboost", "catboost"),
    ("xgboost",  "xgboost"),
    ("jinja2",   "jinja2"),
    ("weasyprint","weasyprint"),
]

print("必須パッケージ:")
for mod, pkg in _REQUIRED:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', '?')
        print(f"  ✓ {pkg:15s} {ver}")
    except ImportError:
        print(f"  ✗ {pkg:15s} pip install {pkg} が必要")

print("\nオプション:")
for mod, pkg in _OPTIONAL:
    try:
        m = importlib.import_module(mod)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  - {pkg} (未インストール・省略可)")

必須パッケージ:


  ✓ lightgbm        4.6.0
  ✓ pandas          2.3.3
  ✓ numpy           2.3.5
  ✓ scikit-learn    1.8.0
  ✓ optuna          4.6.0
  ✓ matplotlib      3.10.0


  ✓ shap            0.50.0

オプション:
  - catboost (未インストール・省略可)
  - xgboost (未インストール・省略可)
  ✓ jinja2
  - weasyprint (未インストール・省略可)


In [4]:
# ── keiba_ai モジュール確認 ──────────────────────────────────
from keiba_ai.constants import FUTURE_FIELDS, UNNECESSARY_COLUMNS
from keiba_ai.feature_engineering import add_derived_features
from keiba_ai.lightgbm_feature_optimizer import LightGBMFeatureOptimizer
from keiba_ai.db_ultimate_loader import load_ultimate_training_frame

print(f"FUTURE_FIELDS:       {len(FUTURE_FIELDS)} 列")
print(f"UNNECESSARY_COLUMNS: {len(UNNECESSARY_COLUMNS)} 列")
print("keiba_ai モジュール: OK")

FUTURE_FIELDS:       30 列
UNNECESSARY_COLUMNS: 188 列
keiba_ai モジュール: OK


In [5]:
# ── DB 確認 ──────────────────────────────────────────────────
s = db_stats(EXISTING_DB)
print("=== 既存 DB ===")
for k, v in s.items():
    print(f"  {k:15s}: {v}")

if DATA_SOURCE_MODE == 'simulation':
    ss = db_stats(SIM_DB)
    print("\n=== Simulation DB ===")
    for k, v in ss.items():
        print(f"  {k:15s}: {v}")

=== 既存 DB ===
  races          : 52450
  results        : 575346
  latest         : 202665060812
  scraped_dates  : 797


In [6]:
# ── 設定を JSON に保存（他 Notebook が読み込み可能）──────────
import json

_cfg = {
    "DATA_SOURCE_MODE": DATA_SOURCE_MODE,
    "FEATURE_SOURCE":   FEATURE_SOURCE,
    "TRAIN_START":      TRAIN_START,
    "TRAIN_END":        TRAIN_END,
    "TEST_START":       TEST_START,
    "TEST_END":         TEST_END,
    "TARGET":           TARGET,
    "CV_FOLDS":         CV_FOLDS,
    "OPTUNA_TRIALS":    OPTUNA_TRIALS,
    "RANDOM_STATE":     RANDOM_STATE,
    "SIM_SCRAPE_START": SIM_SCRAPE_START,
    "SIM_SCRAPE_END":   SIM_SCRAPE_END,
    "updated_at":       datetime.now().isoformat(),
}
_cfg_path = NB_DIR / "config.json"
_cfg_path.write_text(json.dumps(_cfg, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"設定を保存しました: {_cfg_path}")

設定を保存しました: C:\Users\yuki2\Documents\ws\keiba-ai-pro\notebooks\config.json
